In [184]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp

# For visualization
import matplotlib.pyplot as plt


In [170]:

np.random.seed(42)

def get_historic_returns():
    hist_df = pd.read_csv("historic_data_stocks.csv")
    hist_df.rename(columns={"Symbol":"Ticker"}, inplace=True)
    returns_data = hist_df.pivot(index="Date", columns="Ticker", values="return_d5")

    return returns_data

def get_predictions():
    pred_df = pd.read_csv("prob_output_stocks.csv")
    true_df = pd.read_csv("true_labels_stocks.csv")
    pred_df["pred_label"] = np.where(pred_df["Prob"]>0.5, 1, -1)
    pred_df["Date"] = pd.to_datetime(pred_df["Date"])
    true_df["Date"] = pd.to_datetime(true_df["Date"])
    df = pd.merge(true_df, pred_df, on=["Date", "Symbol"], how="inner")
    df = df[['Date', 'Symbol', 'pred_label', "true_label", 'close', 'close_d1', "close_d1_p5"]]
    df.rename(columns={"Symbol":"Ticker"}, inplace=True)
    preds_data = df.pivot(index="Date", columns="Ticker", values="pred_label")
    return preds_data


prediction_data = get_predictions()
five_day_returns = get_historic_returns()
tickers = list(set(np.array(prediction_data.columns)).intersection(set(np.array(five_day_returns.columns))))
prediction_data = prediction_data[tickers]
five_day_returns = five_day_returns[tickers]
mu_5d = five_day_returns.mean()
S_5d = five_day_returns.cov()
S_5d = cp.psd_wrap(S_5d)

def daily_rebalance(mu_5d, S_5d, tickers, predicted_returns):
    adjusted_mu_5d = mu_5d 
    adjusted_mu_5d = adjusted_mu_5d.values
    n = len(tickers)

    w = cp.Variable(n)

    risk_aversion = 0.1
    alpha = 0.1 
    beta = 0

    objective = cp.Maximize(adjusted_mu_5d@w - risk_aversion * cp.quad_form(w, S_5d) - alpha * cp.norm(w, 2) - beta * cp.norm(w, 1))

    # Define the constraints
    constraints = [
        cp.sum(w) == 1,  # Sum of weights equals 1
        w >= -1,        # Weights greater than or equal to -1
        w <= 1,        # Weights less than or equal to 1
        cp.abs(w) <= 0.1
    ]
    # Additional constraints based on predicted returns
    for i, ticker in enumerate(tickers):
        if predicted_returns[ticker] > 0:
            constraints.append(w[i] >= 0)  # Long position
        else:
            constraints.append(w[i] <= 0)  # Short position

    problem = cp.Problem(objective, constraints)
    problem.solve()
    optimal_weights = w.value
    weights_df = pd.DataFrame({'Ticker': tickers, 'Weight': optimal_weights})
    return weights_df

dates = np.array(prediction_data.index)
weights_df = pd.DataFrame([])
for date in dates:
    date_str = pd.to_datetime(date).strftime("%Y_%m_%d")
    prediction = prediction_data.loc[date]
    if len(weights_df)==0:
        weights_df = daily_rebalance(mu_5d, S_5d, tickers, prediction)
        weights_df.rename(columns={"Weight":date_str}, inplace=True)
    else:
        weights_df[date_str] = daily_rebalance(mu_5d, S_5d, tickers, prediction)["Weight"]

weights_df.set_index("Ticker", inplace=True)




2024_03_28
2024_04_01
2024_04_02
2024_04_03
2024_04_04
2024_04_05
2024_04_08
2024_04_09
2024_04_10
2024_04_11
2024_04_12
2024_04_15
2024_04_16
2024_04_17
2024_04_18
2024_04_19
2024_04_22
2024_04_23
2024_04_24
2024_04_25
2024_04_26
2024_04_29
2024_04_30
2024_05_01
2024_05_02
2024_05_03
2024_05_06
2024_05_07
2024_05_08
2024_05_09
2024_05_10
2024_05_13
2024_05_14
2024_05_15
2024_05_16
2024_05_17
2024_05_20
2024_05_21
2024_05_22
2024_05_23


In [171]:
weights_df.describe()

,2024_03_28,2024_04_01,2024_04_02,2024_04_03,2024_04_04,2024_04_05,2024_04_08,2024_04_09,2024_04_10,2024_04_11,...,2024_05_10,2024_05_13,2024_05_14,2024_05_15,2024_05_16,2024_05_17,2024_05_20,2024_05_21,2024_05_22,2024_05_23
count,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,...,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02,4.960000e+02
mean,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,...,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03,2.016129e-03
std,4.568472e-03,4.730263e-03,5.704966e-03,5.804281e-03,5.272548e-03,5.240020e-03,4.984020e-03,5.813341e-03,5.259361e-03,5.549150e-03,...,6.695760e-03,8.559722e-03,5.643166e-03,6.426026e-03,5.712369e-03,6.298269e-03,3.922508e-03,5.869849e-03,4.843607e-03,5.669818e-03
min,-1.133541e-02,-1.291878e-02,-1.348731e-02,-1.679741e-02,-1.425242e-02,-1.307776e-02,-1.345146e-02,-1.515878e-02,-1.505641e-02,-1.418452e-02,...,-1.695799e-02,-2.323785e-02,-1.763179e-02,-2.028490e-02,-1.642068e-02,-1.866207e-02,-9.023296e-03,-1.605157e-02,-1.390707e-02,-1.651243e-02
25%,-2.088985e-11,-6.391887e-10,-7.756054e-10,-5.522377e-10,-3.597244e-10,-6.654799e-11,-1.063575e-10,-4.094427e-10,-1.814683e-10,-5.579443e-10,...,-4.296404e-09,-7.592234e-09,-5.601658e-10,-1.072059e-09,-4.267439e-10,-8.262162e-10,-4.364934e-11,-1.236835e-10,-3.187653e-10,-5.231145e-10
50%,-6.166369e-12,-1.007004e-10,-9.767084e-11,-7.355734e-11,-8.863648e-11,-2.241766e-11,-2.222631e-11,3.443956e-10,-3.089612e-11,-1.712919e-10,...,-3.489606e-11,2.698127e-10,2.514123e-10,6.098755e-10,-5.285556e-11,-1.035387e-10,2.917391e-11,1.980397e-10,-3.431483e-11,-6.258143e-11
75%,2.349969e-03,2.678392e-03,2.306678e-03,3.174334e-03,1.614407e-03,4.580105e-05,2.435412e-03,3.027866e-03,2.120957e-03,1.494125e-03,...,3.917655e-03,3.651546e-03,3.271193e-03,3.327078e-03,2.854966e-03,2.655730e-03,3.528501e-03,3.502522e-03,3.157104e-03,2.597871e-03
max,2.432269e-02,2.724560e-02,4.357164e-02,4.425609e-02,4.187073e-02,4.278900e-02,4.005257e-02,4.285081e-02,4.092013e-02,4.454785e-02,...,4.894381e-02,5.930399e-02,2.911847e-02,4.655577e-02,4.381831e-02,3.291338e-02,2.162006e-02,3.494002e-02,3.191025e-02,4.331036e-02


In [172]:
import numpy as np
import pandas as pd

def compute_portfolio_statistics(weights_df, initial_capital=1000000):
    returns_df =  get_true_returns_for_test_data().T
    returns_df.columns = returns_df.columns.strftime('%Y_%m_%d')

    merge_df = pd.merge(weights_df, returns_df, right_index=True, left_on="Ticker", how="inner", suffixes=('_w', '_r'))
    daily_returns = {}

    for date in weights_df.columns[1:]:
        daily_returns[date] = (merge_df[date+"_w"] * merge_df[date+"_r"]).sum()
    daily_returns= pd.DataFrame.from_dict(daily_returns, orient='index', columns=["Return"])
    # Compute portfolio statistics
    mean_return = daily_returns["Return"].mean()
    volatility = daily_returns["Return"].std()
    sharpe_ratio = mean_return / volatility if volatility != 0 else np.nan
    total_return = daily_returns["Return"].sum()
    total_dollar_return = initial_capital * (1 + total_return)
    
    statistics = {
        'Mean Return': mean_return,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Total Return': total_return,
        'Total Dollar Return': total_dollar_return
    }
    
    return statistics, daily_returns

def get_true_returns_for_test_data():
    pred_df = pd.read_csv("prob_output_stocks.csv")
    true_df = pd.read_csv("true_labels_stocks.csv")
    pred_df["pred_label"] = np.where(pred_df["Prob"]>0.5, 1, -1)
    pred_df["Date"] = pd.to_datetime(pred_df["Date"])
    true_df["Date"] = pd.to_datetime(true_df["Date"])
    df = pd.merge(true_df, pred_df, on=["Date", "Symbol"], how="inner")
    df = df[['Date', 'Symbol', 'pred_label', "true_label", 'close', 'close_d1', "close_d1_p5"]]
    df["return_d5"] = df["close_d1_p5"]/df["close_d1"] - 1
    df.rename(columns={"Symbol":"Ticker"}, inplace=True)
    return_data = df.pivot(index="Date", columns="Ticker", values="return_d5")
    return return_data

stats, daily_returns = compute_portfolio_statistics(weights_df)
print(stats)


{'Mean Return': -0.002287039360888137, 'Volatility': 0.023519023373770643, 'Sharpe Ratio': -0.09724210587071974, 'Total Return': -0.08919453507463734, 'Total Dollar Return': 910805.4649253626}


In [177]:
daily_returns

,Return
2024_04_01,-0.017968
2024_04_02,-0.024298
2024_04_03,0.006002
2024_04_04,-0.003935
2024_04_05,0.003525
2024_04_08,0.027563
2024_04_09,0.005672
2024_04_10,-0.017747
2024_04_11,-0.044113
2024_04_12,-0.036729


In [174]:
import pandas as pd
import numpy as np

def compute_dollar_neutral_weights(weights_df):
    neutral_weights_df = pd.DataFrame(index=weights_df.index, columns=weights_df.columns)
    
    for date in weights_df.columns:
        daily_weights = weights_df[date]
        long_weights = daily_weights[daily_weights >=0]
        short_weights = daily_weights[daily_weights < 0]
        
        long_sum = long_weights.sum()
        short_sum = abs(short_weights.sum())
        print(long_sum, short_sum)
        
        if long_sum == 0 and short_sum == 0:
            # If there are no long or short positions, weights remain the same
            neutral_weights_df.loc[date] = daily_weights
        elif long_sum > 0 and short_sum > 0:
            # print("here")
            # Normalize long and short weights to be dollar-neutral
            long_weights_normalized = long_weights / long_sum
            short_weights_normalized = short_weights / short_sum
            
            neutral_weights_df.loc[long_weights.index, date] = long_weights_normalized 
            neutral_weights_df.loc[short_weights.index, date] = short_weights_normalized 
        else:
            # If only long or only short positions are present
            neutral_weights_df.loc[date] = daily_weights
        
    return neutral_weights_df


neutral_weights_df = compute_dollar_neutral_weights(weights_df)
# print(neutral_weights_df)


1.0829555823927524 0.08295558239275216
1.1256106419445433 0.12561064194454286
1.212229945018179 0.21222994501817924
1.2686929924600523 0.2686929924600534
1.1254965110292297 0.12549651102922993
1.085493122684501 0.0854931226845022
1.129375699259299 0.12937569925929868
1.2673359882589201 0.26733598825892
1.1472997702403886 0.14729977024038884
1.1471807520068384 0.14718075200683794
1.0981253615423494 0.09812536154234802
1.1681429326668455 0.16814293266684543
1.3022938839279181 0.30229388392791656
1.0588899964738103 0.05888999647381196
1.2242213167149136 0.22422131671491558
1.119357189839136 0.1193571898391362
1.2888735319679292 0.2888735319679289
1.1883063477120899 0.18830634771208968
1.1046446360162658 0.10464463601626615
1.0544021113473314 0.0544021113473312
1.4013025398154797 0.40130253981547953
1.2512198761273825 0.25121987612738317
1.0686297178666375 0.06862971786663874
1.1286317382867348 0.12863173828673538
1.1957660970030837 0.195766097003084
1.1403121689633622 0.14031216896336335


In [175]:
neutral_weights_df

,2024_03_28,2024_04_01,2024_04_02,2024_04_03,2024_04_04,2024_04_05,2024_04_08,2024_04_09,2024_04_10,2024_04_11,...,2024_05_10,2024_05_13,2024_05_14,2024_05_15,2024_05_16,2024_05_17,2024_05_20,2024_05_21,2024_05_22,2024_05_23
Ticker,,,,,,,,,,,,,,,,,,,,,
MTD,-0.009744,0.0,0.0,-0.014444,0.0,0.0,-0.016499,-0.019899,-0.021751,0.0,...,-0.012356,-0.0133,-0.016623,-0.016723,-0.015594,-0.014563,-0.022776,-0.018593,0.0,-0.017633
DOC,-0.030936,0.0,0.0,-0.022427,-0.03457,0.0,-0.031042,0.0,0.0,0.0,...,-0.01751,0.0,0.0,-0.023079,0.0,-0.021551,0.0,0.0,-0.031715,-0.027188
RTX,-0.006387,0.0,-0.016204,-0.013109,-0.016432,-0.011071,-0.01413,-0.018521,0.0,0.0,...,0.0,-0.012405,-0.015396,0.0,-0.014125,-0.013315,-0.019799,-0.017402,-0.017312,0.0
O,-0.0,0.0,0.0,0.0,0.0,0.000618,0.0,0.0,-0.008534,-0.0,...,-0.007153,-0.008848,-0.009746,0.0,-0.006734,-0.007561,0.0,0.0,-0.007148,-0.008114
BSX,0.007326,0.006131,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,...,0.004418,-0.0,-0.0,-0.0,-0.0,-0.0,0.005307,-0.0,0.005504,0.005483
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MRNA,0.0,-0.079103,-0.061344,0.0,0.0,0.0,0.0,-0.054959,0.0,0.0,...,-0.03511,-0.032503,0.0,-0.044607,-0.054644,-0.04522,0.0,-0.047137,0.0,0.0
EL,0.0,0.0,-0.06355,-0.051013,0.0,0.0,-0.083039,-0.056703,0.0,-0.076207,...,-0.036269,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.061587
ADM,0.0,0.0,-0.016334,-0.013167,0.0,0.0,-0.014278,-0.018585,0.0,0.0,...,-0.011483,0.0,0.0,-0.01568,0.0,-0.013368,0.0,-0.017474,-0.017417,-0.016086


In [181]:
stats, daily_returns = compute_portfolio_statistics(neutral_weights_df)
stats


{'Mean Return': -0.003939848101454282,
 'Volatility': 0.01820395377648712,
 'Sharpe Ratio': -0.2164281534565931,
 'Total Return': -0.15365407595671698,
 'Total Dollar Return': 846345.924043283}

In [182]:
(daily_returns>0).sum()

Return    20
dtype: int64

In [183]:
daily_returns.describe()

,Return
count,39.000000
mean,-0.003940
std,0.018204
min,-0.044113
25%,-0.014986
50%,0.001206
75%,0.008105
max,0.027563
